# Session 3 — Structured outputs · live demo

Everything on the slides that has code is in this notebook, in the order the
session runs. Nothing here needs a key: it is all `FakeLLM` unless you set a lane
in section 9.

The argument in one line: **the model returns text, and text is untrusted input.**
The parser is where the contract lives, because it is the only part you own.

In [3]:
# Setup. Finds the course from wherever this notebook was opened.
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

# Said here, once, rather than as a NameError four cells down. This notebook has
# to live INSIDE the course checkout: it walks up for pyproject.toml, and if it
# finds none it is sitting somewhere else entirely.
if not CORPUS_DIR.exists():
    raise SystemExit(
        f"No course found above {Path.cwd()}. "
        "Put this notebook inside your dev3pack-cohort-2026-09 checkout "
        "(anywhere in it) and reopen it."
    )
sys.path.insert(0, str(REPO_ROOT / "src"))

from bootcamp_agent.agent import answer_question
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM
from bootcamp_agent.schema import AnswerParseError, parse_research_answer

documents = load_corpus(CORPUS_DIR)
print(f"corpus: {len(documents)} documents from {CORPUS_DIR.name}/")

corpus: 6 documents from corpus/


## 1. The same question, two contracts

One model, two replies. Both are about chunking. Only one of them is data.

In [4]:
question = "How does chunking work?"

prose_llm = FakeLLM(
    default="Chunking is, broadly speaking, quite useful, and many practitioners agree."
)
typed_llm = FakeLLM(
    default=json.dumps(
        {
            "answer": "Chunking splits documents into retrievable passages.",
            "citations": ["rag-basics"],
            "confidence": 0.85,
            "needs_human_review": False,
        }
    )
)

print("PROSE:", prose_llm.complete(system="", user=question))
print()
print("TYPED:", typed_llm.complete(system="", user=question))

PROSE: Chunking is, broadly speaking, quite useful, and many practitioners agree.

TYPED: {"answer": "Chunking splits documents into retrievable passages.", "citations": ["rag-basics"], "confidence": 0.85, "needs_human_review": false}


**Ask the audience to find the confidence in the prose line.** They cannot.

| You need | The prose | The JSON |
|---|---|---|
| How sure is it? | "broadly speaking" | `0.85` — a number a threshold compares |
| Based on what? | "many practitioners" | `["rag-basics"]` — a doc id you can open |
| Does a human need to see this? | nowhere | `false` — a boolean an `if` branches on |

You *could* pull those out of prose with a regex. Then the model rephrases next
week, the regex matches nothing, and downstream code runs on a default. That is
worse than a crash, because nothing tells you.

## 2. Three payloads, three rejections, each naming the field

This is the slide that earns the strictness. Run it and read the three reasons.

In [5]:
for attempt in [
    "The answer is chunking.",
    '{"answer": "x", "citations": []}',
    '{"answer": "x", "citations": [], "confidence": 7, "needs_human_review": false}',
]:
    try:
        parse_research_answer(attempt)
        print("accepted:", attempt[:50])
    except AnswerParseError as error:
        print(f"rejected: {error}")

rejected: Not valid JSON: Expecting value: line 1 column 1 (char 0)
rejected: Wrong fields: missing=['confidence', 'needs_human_review'] unknown=[]
rejected: 'confidence' out of range [0, 1]: 7


Three different reasons, each naming the field. `"invalid response"` would have
told you nothing at 2am.

The third one is the one worth pausing on: the model read `confidence` as *out of
ten*. Nothing about `7` is malformed — it is the wrong scale, and only a range
check catches it.

## 3. Why a *helpful* extra field is still a rejection

The threat is not a hostile model. It is a helpful one.

In [6]:
helpful = json.dumps(
    {
        "answer": "Chunking splits documents into retrievable passages.",
        "citations": ["rag-basics"],
        "confidence": 0.85,
        "needs_human_review": False,
        "source_url": "https://example.com/chunking",   # nobody asked for this
    }
)

try:
    parse_research_answer(helpful)
except AnswerParseError as error:
    print(f"rejected: {error}")

rejected: Wrong fields: missing=[] unknown=['source_url']


The gate is set **equality**, not a subset test, and it is the choice people
argue with. Accept the extra field and three things follow:

1. `ResearchAnswer` no longer describes what the object holds, so the next reader guesses.
2. It appears on some replies and not others, so any code using it needs a fallback nobody tests.
3. A field you never asked for is a field nobody validates.

One comparison catches a missing field **and** an unknown one.

## 4. The limit of the shape

Every gate passes. The content is nonsense.

In [7]:
cheese = json.dumps(
    {
        "answer": "The moon is cheese.",
        "citations": ["rag-basics"],
        "confidence": 0.9,
        "needs_human_review": False,
    }
)

accepted = parse_research_answer(cheese)
print("parser said: ACCEPTED")
print(f"  answer     = {accepted.answer!r}")
print(f"  confidence = {accepted.confidence}")
print()
print("Schema conformance is necessary. It is never sufficient.")

parser said: ACCEPTED
  answer     = 'The moon is cheese.'
  confidence = 0.9

Schema conformance is necessary. It is never sufficient.


So two more checks sit on top of the shape, both in `agent.py`, both visible in
the trace — and sections 5 and 6 are those two.

## 5. The application notices what the model made up

The model cites a document. The retriever never returned it. Watch who catches it.

In [8]:
fabricator = FakeLLM(
    default=json.dumps(
        {
            "answer": "Chunking splits documents into retrievable passages.",
            "citations": ["rag-basics", "internal-wiki"],   # one of these is invented
            "confidence": 0.95,
            "needs_human_review": False,
        }
    )
)

result = answer_question("How does chunking work?", documents, fabricator)
for event in result.trace:
    print(f"trace[{event.kind}] {event.detail}")

print()
print(f"citations  = {list(result.answer.citations)}")
print(f"confidence = {result.answer.confidence}")
print(f"review     = {result.answer.needs_human_review}")

trace[retrieve] top_k=3 -> [('evaluation-basics', 0), ('prompt-injection', 2), ('rag-basics', 1)]
trace[llm_call] attempt 1: 161 chars
trace[decision] fabricated citations stripped: ['internal-wiki']; flagged for human review

citations  = ['rag-basics']
confidence = 0.2
review     = True


`internal-wiki` is stripped, confidence is capped, and the review flag goes true.

**The application, not the model, is the one that noticed.** You cannot ask the
thing you are checking whether it is telling the truth.

## 6. Refusing before spending a call

Retrieval runs first. Nothing relevant means no model call at all — and the
cheapest call is the one you do not make.

In [9]:
watchful = FakeLLM(default=json.dumps({
    "answer": "I would have answered.", "citations": [],
    "confidence": 0.9, "needs_human_review": False,
}))

result = answer_question("How do I repot an orchid?", documents, watchful)
for event in result.trace:
    print(f"trace[{event.kind}] {event.detail}")

print()
print(f"model calls made: {len(watchful.calls)}")
print(f"citations={list(result.answer.citations)} review={result.answer.needs_human_review}")

trace[retrieve] top_k=3 -> []
trace[decision] no relevant chunks; refusing without an LLM call

model calls made: 0
citations=[] review=True


Read the trace out loud: `retrieve` ran, `decision` refused, and there is **no
`llm_call` line at all**. `watchful.calls` is empty — the fake would happily have
answered, and was never asked.

> **This is the floor, not your answer.** `ch03-e2` asks you for an *unsupported*
> question of your own. The check accepts anything that retrieves nothing, so
> copying this one passes and teaches you nothing. The interesting part is finding
> a question a person would really ask that still leaves the corpus — every word
> that also appears in a document drags a chunk back.

## 7. One retry, then a typed refusal

The budget is two calls. Never an unbounded loop. This model never complies.

In [10]:
stubborn = FakeLLM(default="Of course! Here is the JSON you asked for:")

result = answer_question("How does chunking work?", documents, stubborn)
for event in result.trace:
    print(f"trace[{event.kind}] {event.detail}")

print()
print(f"model calls made: {len(stubborn.calls)}")
print(f"citations={list(result.answer.citations)} review={result.answer.needs_human_review}")
print()
print("Nothing raised. The flag and the two llm_call lines are the only evidence,")
print("which is exactly why both of them exist.")

trace[retrieve] top_k=3 -> [('evaluation-basics', 0), ('prompt-injection', 2), ('rag-basics', 1)]
trace[llm_call] attempt 1: 42 chars
trace[decision] parse failed (Not valid JSON: Expecting value: line 1 column 1 (char 0)); retrying once
trace[llm_call] attempt 2: 42 chars
trace[decision] parse failed twice (Not valid JSON: Expecting value: line 1 column 1 (char 0)); flagged refusal

model calls made: 2
citations=[] review=True

Nothing raised. The flag and the two llm_call lines are the only evidence,
which is exactly why both of them exist.


## 8. The happy path, so they have seen one

Same agent, a model that complies.

In [11]:
good = FakeLLM(default=json.dumps({
    "answer": "Chunking splits documents into passages small enough to retrieve.",
    "citations": ["rag-basics"],
    "confidence": 0.8,
    "needs_human_review": False,
}))

result = answer_question("How does chunking work?", documents, good)
for event in result.trace:
    print(f"trace[{event.kind}] {event.detail}")

print()
print(result.answer.answer)
print(f"citations={list(result.answer.citations)} confidence={result.answer.confidence}")

trace[retrieve] top_k=3 -> [('evaluation-basics', 0), ('prompt-injection', 2), ('rag-basics', 1)]
trace[llm_call] attempt 1: 156 chars
trace[decision] answered with citations ['rag-basics']

Chunking splits documents into passages small enough to retrieve.
citations=['rag-basics'] confidence=0.8


## 9. The same agent, on a real lane

Set `BOOTCAMP_PROVIDER` in `.env` — `anthropic`, `openai`, or `ollama` for a local
model. With nothing set it resolves to the **fake** lane and says so, and the demo
carries on: **nothing here needs a key.**

Read the first line of the output before you read anything else. It names the lane
you are actually on, which is the one thing worth being sure about in front of a
room.

In [12]:
try:
    from bootcamp_agent.config import load_settings
    from bootcamp_agent.llm import get_client

    settings = load_settings()
    live = get_client(settings)
    print(f"lane: {settings.provider} · {settings.model}\n")

    result = answer_question("How does chunking work?", documents, live)
    for event in result.trace:
        print(f"trace[{event.kind}] {event.detail}")
    print()
    print(result.answer.answer)
    print(f"citations={list(result.answer.citations)} "
          f"confidence={result.answer.confidence}")
except Exception as error:
    print(f"no live lane ({type(error).__name__}: {error})")
    print("Everything above ran on FakeLLM and is unaffected.")

lane: ollama · None

trace[retrieve] top_k=3 -> [('evaluation-basics', 0), ('prompt-injection', 2), ('rag-basics', 1)]
trace[llm_call] attempt 1: 260 chars
trace[decision] answered with citations ['rag-basics']

Chunking splits documents into passages small enough to be individually relevant — respecting paragraph boundaries beats cutting at a fixed character count mid-sentence.
citations=['rag-basics'] confidence=1.0


Whatever the model says, it comes back through the same parser. That is the
whole point of the seam: **the contract does not get weaker because the model got
better.**

## What to take away

- The model returns **text**. Text is untrusted input, including from a model you pay for.
- A provider's structured-output mode is a claim made by the thing you are checking.
- The parser is strict and loud: every rejection names the field.
- Set equality, not a subset — a *helpful* extra field is still a rejection.
- Valid JSON can be a wrong answer, so citations are verified against retrieval.
- Retrieval refuses before a call is spent, and the repair budget is two calls, then a typed refusal.

Session 4 gives the model **tools**. Every argument it passes needs exactly the
treatment you just gave its output.